# Basic Usage of `fvs2py`
This notebook gives a basic overview of how `fvs2py` allows you to load a regional FVS library, load a keyfile with instructions for an FVS simulation, and then execute a simulation. 

Please note that fvs2py is currently still in early development and the API may undergo breaking changes as common use cases and patterns emerge more clearly with increasing use of the package. 

In [1]:
import importlib.resources
import os
from tempfile import TemporaryDirectory

from fvs2py import FVS
from fvs2py.enums import FvsVariant

## Load Regional FVS Library
Instantiate FVS using a compiled version of the regional library you want to use. We're operating in Linux environment, and have all US variants built with Shared Object ("*.so") files saved in `/usr/local/lib`.

In this example, we load the FVS Inland Empire variant library.

In [2]:
variant = FvsVariant.INLAND_EMPIRE

In [3]:
fvs = FVS(f"/usr/local/lib/FVS{variant.lower()}.so")

FVS keeps track of its run state using an internal return code, which you can access as the `itrncd` property of the FVS instance.

In [4]:
# let's get an explanation of the codes from the docstring
fvs.itrncd?

Type:        property
String form: <property object at 0x7336a1f4d1c0>
Docstring:  
Returns with the current return code value in FVS.

-1: indicates that FVS has not been started.
 0: indicates that FVS is in good running state.
 1: indicates that FVS has detected an error of some kind and should not
        be used until reset by specifying new input.
 2: indicates that FVS has finished processing all the stands; new input
        can be specified.

In [5]:
# let's check the current state
fvs.itrncd  # should be -1

-1

Some information related to the regional variant is now defined.

In [6]:
fvs.variant

'IE'

In [7]:
fvs.dims

{'ntrees': 0,
 'ncycles': 0,
 'nplots': 0,
 'maxtrees': 3000,
 'maxspecies': 23,
 'maxplots': 500,
 'maxcycles': 40}

In [8]:
fvs.species_codes

,fvs_index,fvs_alpha,fia,plants
0,1,WP,119,PIMO3
1,2,WL,073,LAOC
2,3,DF,202,PSME
3,4,GF,017,ABGR
4,5,WH,263,TSHE
5,6,RC,242,THPL
6,7,LP,108,PICO
7,8,ES,093,PIEN
8,9,AF,019,ABLA
9,10,PP,122,PIPO


## Load a Keyfile
To run FVS, you need to load a keyfile with instructions for the simulation. `fvs2py` does not provide helpers for constructing keyfiles. We have several simple keyfiles used for testing in this repo and will use one of those as an example here. 

Once you start simulations, FVS will generate artifacts (e.g., an output file (\*.out), treelist file (\*.trl), or output database (*.db)) depending on the outputs specified in the keyfile. To avoid cluttering up the directory where we keep our test keyfiles, we'll create a temporary directory to work in a make copy of our keyfile in there.

In [9]:
temp_dir = TemporaryDirectory()
os.chdir(temp_dir.name)
keyfile_content = (
    importlib.resources
    .files("fvs2py.tests.keyfiles")
    .joinpath(f"{variant}.key")
    .read_text()
)
with open(f"{temp_dir.name}/example_keyfile.key", "w") as f:
    f.write(keyfile_content)
keyfile_to_load = f"{temp_dir.name}/example_keyfile.key"

In [10]:
# load the keyfile to initialize an FVS simulation
fvs.load_keyfile(keyfile_to_load)

In [11]:
# you can see the contents of the keyfile using the `keyfile` attribute
print(fvs.keyfile)

STDIDENT
12345 TEST
STDINFO          118       260         1
DESIGN            -1         1
INVYEAR       1990.0
NUMCYCLE        10.0
TREEFMT
(I4,I4,F6.0,I1,A3,F4.1,F3.1,2F3.0,F4.1,I1,
6I2,2I1,I2,2I3,2I1,F3.0)
TREEDATA        15.0
0101 100    501  1 51
0101 101    501  2 41
-999
PROCESS
STOP


In [12]:
# the internal return code gets updated in-place
fvs.itrncd  # should now be 0

0

In [13]:
# until you start running a keyfile, information about the stand and simulation settings will be undefined
type(fvs.summary)  # should be None

NoneType

Running a single-stand keyfile is a single call: `fvs.run()` drives 
FVS through every cycle of the stand and then issues the final flush 
of the main output file before returning. Once it returns, `fvs.itrncd` 
is `2` (`FINISHED_ALL_STANDS`) and the simulation outputs are ready to 
inspect.

For keyfiles that describe more than one stand, `FVS.run_batch()` 
offers a lower-level entry point that advances one stand at a time 
without auto-flushing, so callers can drive the multi-stand loop 
themselves. Using it requires loading the keyfile with 
`check_single_stand=False`. The rest of this notebook sticks to the 
single-stand case via `run()`.

In [14]:
fvs.run?

Signature: fvs.run(stop_point_code: 'int' = 0, stop_point_year: 'int' = 0) -> 'None'
Docstring:
Run a single-stand FVS simulation to completion.

The call stays eager: when the stand finishes (FVS reports
``restart_code == FvsRestartCode.DONE_RUNNING_STAND``), :meth:`run`
issues one additional ``_fvs`` call to flush the main output file
before returning, so callers do not need to invoke :meth:`run` a
second time just to finalize output. When ``stop_point_code`` pauses
the simulation mid-cycle, no flush is performed and the next call to
:meth:`run` resumes from the stop.

Multi-stand keyfiles are out of scope here; use :meth:`run_batch`
(and load the keyfile with ``check_single_stand=False``) when the
keyfile defines more than one stand.

Args:
    stop_point_code (optional, int): when FVS should stop during a cycle:
       -1 : Stop at every stop location
        0 : Never stop
        1 : Stop just before the first call to the Event Monitor
        2 : Stop just after the first call t

In [15]:
fvs.run()

`fvs.restart_code` reports where FVS paused inside a cycle; it is 
mostly interesting when `run()` is called with a non-zero 
`stop_point_code`. After a completed single-stand `run()` it resets to 
`0` (`INITIALIZED`).

In [16]:
fvs.restart_code?

Type:        property
String form: <property object at 0x7336986ed8a0>
Docstring:  
A code indicating when FVS stopped.

  1: Stop was done just before the first call to the Event Monitor.
  2: Stop was done just after the first call to the Event Monitor.
  3: Stop was done just before the second call to the Event Monitor.
  4: Stop was done just after the second call to the Event Monitor.
  5: Stop was done after growth and mortality has been computed, but
        prior to applying them.
  6: Stop was done just before the ESTAB routines are called.
100: Stop was done after a stand has been simulated but prior to
        starting a subsequent stand.

In [17]:
# run() has completed the stand and flushed outputs: itrncd == 2 and restart_code == 0
fvs.itrncd, fvs.restart_code

(2, 0)

Now the outputs generated by FVS should be complete, and we can inspect outputs like the summary table or species attributes, which are returned as Pandas DataFrames.

In [18]:
fvs.summary

,year,age,tpa,tcuft,mcuft,bdft,rtpa,rtcuft,rmcuft,rbdft,atba,atccf,attopht,prdlen,acc,mort,sampwt,fortyp,sizecls,stkcls
0,1990,1,100,145,0,0,0,0,0,0,12,16,30,10,28,0,1,999,5,5
1,2000,11,97,420,213,1070,0,0,0,0,26,31,40,10,47,1,1,999,5,5
2,2010,21,172,876,701,3247,0,0,0,0,44,49,46,10,68,3,1,996,1,4
3,2020,31,141,1525,1289,6234,0,0,0,0,65,68,55,10,62,5,1,996,1,4
4,2030,41,154,2091,1822,8527,0,0,0,0,81,81,61,10,88,12,1,996,1,4
5,2040,51,129,2845,2494,12863,0,0,0,0,100,96,68,10,90,16,1,996,1,4
6,2050,61,197,3592,3213,16503,0,0,0,0,118,109,73,10,90,19,1,996,1,4
7,2060,71,149,4296,3849,20724,0,0,0,0,133,120,78,10,115,30,1,996,1,3
8,2070,81,158,5144,4639,26051,0,0,0,0,149,132,83,10,99,29,1,996,1,3
9,2080,91,123,5845,5347,30507,0,0,0,0,161,140,86,10,96,46,1,996,1,3


The `species` attribute contains a mixture of things, including the species codes as well as parameters that may be static (fixed over simulation) such as merchantability specifications (e.g., bfmind, bfstmp, etc.) as well as dynamic values that evolve over the course of a simulation (e.g., spccf).

In [19]:
fvs.species

,fvs_index,fvs_alpha,fia,plants,spccf,spsdi,spsiteindx,bfmind,bftopd,bfstmp,...,mctopd,mcstmp,mcmeth,baimult,htgmult,mortmult,mortdia1,mortdia2,regdmult,reghmult
0,1,WP,119,PIMO3,0.000000,668.675415,0.000000,7.0,4.5,1.0,...,4.5,1.0,6.0,1.0,1.0,1.0,0.0,99999.0,1.0,1.0
1,2,WL,073,LAOC,0.007875,668.675415,0.000000,7.0,4.5,1.0,...,4.5,1.0,6.0,1.0,1.0,1.0,0.0,99999.0,1.0,1.0
2,3,DF,202,PSME,5.413642,668.675415,0.000000,7.0,4.5,1.0,...,4.5,1.0,6.0,1.0,1.0,1.0,0.0,99999.0,1.0,1.0
3,4,GF,017,ABGR,0.000000,668.675415,0.000000,7.0,4.5,1.0,...,4.5,1.0,6.0,1.0,1.0,1.0,0.0,99999.0,1.0,1.0
4,5,WH,263,TSHE,0.000000,668.675415,0.000000,7.0,4.5,1.0,...,4.5,1.0,6.0,1.0,1.0,1.0,0.0,99999.0,1.0,1.0
5,6,RC,242,THPL,0.000000,668.675415,0.000000,7.0,4.5,1.0,...,4.5,1.0,6.0,1.0,1.0,1.0,0.0,99999.0,1.0,1.0
6,7,LP,108,PICO,0.021423,668.675415,0.000000,6.0,4.5,1.0,...,4.5,1.0,6.0,1.0,1.0,1.0,0.0,99999.0,1.0,1.0
7,8,ES,093,PIEN,0.000000,668.675415,0.000000,7.0,4.5,1.0,...,4.5,1.0,6.0,1.0,1.0,1.0,0.0,99999.0,1.0,1.0
8,9,AF,019,ABLA,0.000000,668.675415,0.000000,7.0,4.5,1.0,...,4.5,1.0,6.0,1.0,1.0,1.0,0.0,99999.0,1.0,1.0
9,10,PP,122,PIPO,11.501995,668.675415,0.000000,7.0,4.5,1.0,...,4.5,1.0,6.0,1.0,1.0,1.0,0.0,99999.0,1.0,1.0


FVS has probably generated some artifacts in the same directory as the keyfile being run. Note that you may see FVS generate an output database in the current working directory, which may be different from the location where the keyfile exists. We changed to the temp directory before running FVS to avoid having it create FVSOut.db in the same directory as this notebook.

In [20]:
os.listdir(temp_dir.name)

['example_keyfile.key', 'FVSOut.db', 'example_keyfile.out']

In [21]:
with open(keyfile_to_load.replace(".key", ".out")) as outfile:
    print(outfile.read())



     FOREST VEGETATION SIMULATOR     VERSION FS2025.4 -- INLAND EMPIRE EXPANDED                 RV:20250930    04-19-2026  02:33:48

----------------------------------------------------------------------------------------------------------------------------------

                                                OPTIONS SELECTED BY INPUT

KEYWORD FILE NAME: /tmp/tmp8932hdi7/example_keyfile
----------------------------------------------------------------------------------------------------------------------------------
KEYWORD    PARAMETERS:
--------   -----------------------------------------------------------------------------------------------------------------------

STDIDENT
           STAND ID= 12345                             TEST                                                                    

           HABITAT TYPE WILL BE MAPPED TO 260 FOR THIS PROJECTION.

STDINFO    FOREST-LOCATION CODE=     118; HABITAT TYPE=260; AGE=    1; ASPECT AZIMUTH IN DEGREES=   0.; SLOPE=   5

## Watching the trees grow.
You can inspect the outputs mid-simulaton by calling `run` and providing values for `stop_point_code` and/or `stop_point_year`. Here we show an example stopping each cycle after growth and mortality have been calculated, but before any treatments have been triggered. 

Be conscious of the fact that `fvs.summary` and all other attributes of an FVS instance get mutated in-place over the course of the simulation by FVS. 

You may also notice that "age" and "after treatment" columns (e.g., `atba` and `attopht` in the example below) are not finalized until the end of a cycle, and thus may not be accurate when you first are able to view them in the `fvs.summary` attribute during a simulation. There is not currently a way to stop FVS at the point in a cycle after treatments have been simulated but before the next cycle begins.

In [22]:
fvs.load_keyfile(keyfile_to_load)
i = 1
while fvs.itrncd < 2:  # run until the simulation is done
    fvs.run(
        stop_point_code=6, stop_point_year=-1
    )  # stop at beginning of each each cycle before treatments
    print(f"Run called {i} time(s): ({fvs.itrncd=}; {fvs.restart_code=})")
    display(fvs.summary)
    i += 1

Run called 1 time(s): (fvs.itrncd=0; fvs.restart_code=6)


,year,age,tpa,tcuft,mcuft,bdft,rtpa,rtcuft,rmcuft,rbdft,atba,atccf,attopht,prdlen,acc,mort,sampwt,fortyp,sizecls,stkcls
0,1990,0,100,145,0,0,0,0,0,0,0,0,0,0,0,0,0,999,5,5


Run called 2 time(s): (fvs.itrncd=0; fvs.restart_code=6)


,year,age,tpa,tcuft,mcuft,bdft,rtpa,rtcuft,rmcuft,rbdft,atba,atccf,attopht,prdlen,acc,mort,sampwt,fortyp,sizecls,stkcls
0,1990,1,100,145,0,0,0,0,0,0,12,16,30,10,28,0,1,999,5,5
1,2000,0,97,420,213,1070,0,0,0,0,0,0,0,0,0,0,0,999,5,5


Run called 3 time(s): (fvs.itrncd=0; fvs.restart_code=6)


...

,year,age,tpa,tcuft,mcuft,bdft,rtpa,rtcuft,rmcuft,rbdft,atba,atccf,attopht,prdlen,acc,mort,sampwt,fortyp,sizecls,stkcls
0,1990,1,100,145,0,0,0,0,0,0,12,16,30,10,28,0,1,999,5,5
1,2000,11,97,420,213,1070,0,0,0,0,26,31,40,10,47,1,1,999,5,5
2,2010,21,172,876,701,3247,0,0,0,0,44,49,46,10,68,3,1,996,1,4
3,2020,31,141,1525,1289,6234,0,0,0,0,65,68,55,10,62,5,1,996,1,4
4,2030,41,154,2091,1822,8527,0,0,0,0,81,81,61,10,88,12,1,996,1,4
5,2040,51,129,2845,2494,12863,0,0,0,0,100,96,68,10,90,16,1,996,1,4
6,2050,61,197,3592,3213,16503,0,0,0,0,118,109,73,10,90,19,1,996,1,4
7,2060,71,149,4296,3849,20724,0,0,0,0,133,120,78,10,115,30,1,996,1,3
8,2070,0,158,5144,4639,26051,0,0,0,0,0,0,0,0,0,0,0,996,1,3


Run called 10 time(s): (fvs.itrncd=0; fvs.restart_code=6)


,year,age,tpa,tcuft,mcuft,bdft,rtpa,rtcuft,rmcuft,rbdft,atba,atccf,attopht,prdlen,acc,mort,sampwt,fortyp,sizecls,stkcls
0,1990,1,100,145,0,0,0,0,0,0,12,16,30,10,28,0,1,999,5,5
1,2000,11,97,420,213,1070,0,0,0,0,26,31,40,10,47,1,1,999,5,5
2,2010,21,172,876,701,3247,0,0,0,0,44,49,46,10,68,3,1,996,1,4
3,2020,31,141,1525,1289,6234,0,0,0,0,65,68,55,10,62,5,1,996,1,4
4,2030,41,154,2091,1822,8527,0,0,0,0,81,81,61,10,88,12,1,996,1,4
5,2040,51,129,2845,2494,12863,0,0,0,0,100,96,68,10,90,16,1,996,1,4
6,2050,61,197,3592,3213,16503,0,0,0,0,118,109,73,10,90,19,1,996,1,4
7,2060,71,149,4296,3849,20724,0,0,0,0,133,120,78,10,115,30,1,996,1,3
8,2070,81,158,5144,4639,26051,0,0,0,0,149,132,83,10,99,29,1,996,1,3
9,2080,0,123,5845,5347,30507,0,0,0,0,0,0,0,0,0,0,0,996,1,3


Run called 11 time(s): (fvs.itrncd=2; fvs.restart_code=0)


,year,age,tpa,tcuft,mcuft,bdft,rtpa,rtcuft,rmcuft,rbdft,atba,atccf,attopht,prdlen,acc,mort,sampwt,fortyp,sizecls,stkcls
0,1990,1,100,145,0,0,0,0,0,0,12,16,30,10,28,0,1,999,5,5
1,2000,11,97,420,213,1070,0,0,0,0,26,31,40,10,47,1,1,999,5,5
2,2010,21,172,876,701,3247,0,0,0,0,44,49,46,10,68,3,1,996,1,4
3,2020,31,141,1525,1289,6234,0,0,0,0,65,68,55,10,62,5,1,996,1,4
4,2030,41,154,2091,1822,8527,0,0,0,0,81,81,61,10,88,12,1,996,1,4
5,2040,51,129,2845,2494,12863,0,0,0,0,100,96,68,10,90,16,1,996,1,4
6,2050,61,197,3592,3213,16503,0,0,0,0,118,109,73,10,90,19,1,996,1,4
7,2060,71,149,4296,3849,20724,0,0,0,0,133,120,78,10,115,30,1,996,1,3
8,2070,81,158,5144,4639,26051,0,0,0,0,149,132,83,10,99,29,1,996,1,3
9,2080,91,123,5845,5347,30507,0,0,0,0,161,140,86,10,96,46,1,996,1,3


Let's clean up after ourselves.

In [23]:
temp_dir.cleanup()